# GEMS P6 — Boost Standalone | Colab CPU

**Independent runner for P6 Boost r2**, reusing existing P6 W2 artifacts from `MyDrive/GEMS_2`. It does **not** rerun W0–W2, G01, G02 full research, M9 training, or W3.

**Execution:** Config → Mount & clone → Startup verification → Restore context → Stage W2 → Preflight → A → optional G02 backfill → Preflight again → B → C → D → E → Report.

First run: only `preflight` and `benchmark` enabled. No submission is uploaded by code. Stage E export defaults OFF. Source code is loaded from the selected Git ref; branch `p6-boost` must be pushed first or use a verified commit SHA on `main`.

## 0 — Controls (edit before running)

In [ ]:
CONFIG = {
    'repo': 'mybenkhadda/GEMS---drivendata',
    'ref': 'main',  # CHANGE TO 'p6-boost' after pushing r2, or preferably pin its commit SHA
    'drive_root': '/content/drive/MyDrive/GEMS_2',
    'local_root': '/content/gems_work',
    'run_mode': 'PREPARE_DATA',
    'failure_policy': 'interactive',
    'download_missing': False,
    'run_advanced_models': False,
    'train_baselines': [],
    'run_full_inference': False,
    'run_submission': False,
    'feature_stage': 'lazy',
    'use_gpu': False,
    'seed': 2026,
}
PRIVATE_REPO = False  # if True, Colab Secrets must contain GITHUB_TOKEN
DEM_CONFIG = {'enabled': False, 'mode': 'inventory_only', 'download': False}
M9_CONFIG = {'profile': 'PILOT', 'overwrite_existing': False}
P6_CONFIG = {'overwrite_existing': False}
BOOST_CONFIG = {'geology': {'backfill_missing': False}}
BOOST_RUN = {'preflight': True, 'benchmark': True, 'backfill_g02': False,
             'geology': False, 'robustness': False, 'hybrid': False,
             'selection': False, 'export': False, 'report': True}
EXPECT = {'notebook_version': 'p6-nb.3', 'code_rev': 'p6b.1-r2', 'commit': None}
# IMPORTANT: For the latest r2 workflow, CONFIG['ref'] must resolve to a commit with boost_audit.py.


## 1 — Google Drive, repository clone, version lock

Requires the exact P6 r2 implementation in the selected Git ref. A missing branch is an error. Do not silently use `main` if you selected `p6-boost`.

In [ ]:
import base64, os, subprocess, sys
from pathlib import Path
from google.colab import drive
drive.mount('/content/drive')
CODE = '/content/GEMS'
REPO_URL = f"https://github.com/{CONFIG['repo']}.git"

def _git_env():
    if not PRIVATE_REPO:
        return None
    from google.colab import userdata
    token = userdata.get('GITHUB_TOKEN') or ''
    if not token:
        raise RuntimeError('PRIVATE_REPO=True but GITHUB_TOKEN is missing in Colab Secrets')
    hdr = base64.b64encode(f'x-access-token:{token}'.encode()).decode()
    return {**os.environ, 'GIT_CONFIG_COUNT': '1', 'GIT_CONFIG_KEY_0': 'http.https://github.com/.extraheader',
            'GIT_CONFIG_VALUE_0': f'AUTHORIZATION: basic {hdr}'}

def git(*args, auth=False, check=True):
    r = subprocess.run(['git', *args], capture_output=True, text=True,
                       env=_git_env() if auth else None)
    if check and r.returncode:
        raise RuntimeError(f"Git failed: {' '.join(args[:4])}\n{r.stderr[-1500:]}")
    return r

if not Path(CODE, '.git').exists():
    git('clone', '-q', REPO_URL, CODE, auth=True)
git('-C', CODE, 'remote', 'set-url', 'origin', REPO_URL)
git('-C', CODE, 'fetch', '-q', '--tags', 'origin', auth=True)
ref = CONFIG['ref']
remote_branch = git('-C', CODE, 'rev-parse', '--verify', '-q', f'refs/remotes/origin/{ref}', check=False).returncode == 0
if remote_branch:
    git('-C', CODE, 'checkout', '-q', '-B', 'p6_colab_active', f'origin/{ref}')
else:
    # For a SHA/tag, require Git to resolve it; do not quietly switch to main.
    git('-C', CODE, 'checkout', '-q', '--detach', ref)
HEAD = git('-C', CODE, 'rev-parse', 'HEAD').stdout.strip()
if 'gems_bench' in sys.modules:
    raise RuntimeError('gems_bench already imported. Restart runtime to avoid stale module code.')
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '-e', f'{CODE}[colab]'], check=True)
if CODE not in sys.path:
    sys.path.insert(0, CODE)
os.environ['GEMS_CODE_COMMIT'] = HEAD
print('Requested:', ref, '| Resolved SHA:', HEAD)


## 2 — Startup contract

This deliberately fails with older `main` code that lacks r2. Push the latest implementation or select its exact commit before proceeding.

In [ ]:
import json
from IPython.display import display
import pandas as pd
from gems_bench.p6.boost_audit import startup_check
START = startup_check(CONFIG['ref'], HEAD,
    {**EXPECT, 'code_root': CODE,
     'artifact_root': f"{CONFIG['drive_root']}/outputs/p6_fast_track/boost"}, BOOST_CONFIG)
print(json.dumps(START, indent=2, default=str))


## 3 — Reattach existing M9/P6 context (no retraining)

In [ ]:
from gems_bench.pipeline import Pipeline
from gems_bench.p6 import P6FastTrack

pipe = Pipeline.from_notebook(CONFIG, DEM_CONFIG)
pipe.step('bootstrap')
for step in ('raw_data', 'native_grid', 'spatial_folds', 'discovery_splits', 'feature_store'):
    pipe.step(step)  # validate + stage cached Drive data, never rebuild
m9 = pipe.m9(M9_CONFIG)
m9.load()
p6 = P6FastTrack(m9, P6_CONFIG)
print('P6 workspace:', p6.ws.root)
display(p6.budget.table())


## 4 — Restore frozen W2 inputs

A–E depend on the *existing* W2 candidate JSON, sampled rows and 6 stored prediction maps. Missing inputs stop here: do **not** regenerate W2 in this notebook.

In [ ]:
bst = p6.boost(BOOST_CONFIG)
need = bst.stage()
required = ['stack/candidate.json', 'stack/features/rows.parquet']
missing = [name for name in required if not need.get(name)]
if need.get('stack/maps/*.npy', 0) != 6:
    missing.append(f"stack/maps/*.npy expected 6, got {need.get('stack/maps/*.npy', 0)}")
if missing:
    raise RuntimeError('Frozen W2 artifacts missing: ' + ', '.join(missing) +
                       '. Restore P6 W2 artifacts on Drive; do not rerun W2.')
print('Restored:', need)
print('Champion:', json.dumps(bst.champion(), indent=2, default=str))


## 5 — Preflight (run before A; rerun after any G02 backfill)

Expected known issues: G02 zone-2 PARTIAL; W2 submitted representation was skeleton@5%, not its nested discovery choice. Investigate BLOCKED evaluation contracts before selection.

In [ ]:
if BOOST_RUN['preflight']:
    pre = bst.preflight()
    print(json.dumps(pre['areas'], indent=2, default=str))
    display(bst.load_table('feature_compatibility', 'preflight'))
    sp = (bst.read_json('preflight', 'submission_provenance') or {}).get('w2', {})
    print('W2 submission provenance:', json.dumps({k: sp.get(k) for k in
          ('status', 'rep', 'frac', 'blend', 'selected_cell', 'interpretation')}, indent=2, default=str))
    print('G02 backfill plan:', json.dumps(bst.read_json('preflight','g02_backfill_plan'), indent=2, default=str)[:4000])


## 6 — A: Matched ranking × representation × budget

Read `boost/benchmark/verdict.json` first. Decide whether observed W2 advantage comes from ranking or representation/budget. No new DTI is claimed until this cell runs on the real data.

In [ ]:
if BOOST_RUN['benchmark']:
    a = bst.a_benchmark()
    display(a['ranking_gain'])
    display(a['nested'][['space','disc_dti','disc_worst','geo_dti','systems_gained','systems_lost']])
    print('VERDICT:', json.dumps(a['verdict'], indent=2, default=str))


## 7 — B0: Optional G02 zone-2 backfill

**OFF by default.** Inspect `preflight/g02_backfill_plan.json`, validate original-input compatibility and remaining budget first. Backfill is an explicit heavy computation; run preflight again afterward.

In [ ]:
if BOOST_RUN['backfill_g02']:
    plan = bst.read_json('preflight','g02_backfill_plan') or {}
    assert plan.get('status') in ('PENDING','READY'), (
        f"Backfill not permitted: {plan.get('status')} — {plan.get('reason') or plan.get('input_drift')}")
    BOOST_CONFIG = {**BOOST_CONFIG, 'geology': {**BOOST_CONFIG.get('geology', {}),
                                               'backfill_missing': True}}
    bst = p6.boost(BOOST_CONFIG)
    completed = bst.section('geology_backfill', bst.backfill)
    print('Backfill result:', completed)
    print('Re-run cell 5 (preflight) before B.')


## 8 — B: G02-C / G02-A / G01 geological ablation

S0–S3 reuse W2 sampled rows. If any required zone raster is absent, the affected variant must be BLOCKED, not evaluated on fewer zones.

In [ ]:
if BOOST_RUN['geology']:
    b = bst.b_geology()
    print(json.dumps({k: v for k, v in b['attach'].items() if k != 'root_cause'}, indent=2, default=str))
    display(b['audit'].pivot_table(index='unit', columns='feature', values='key_present', aggfunc='first'))
    display(b['ablation'][['candidate','status','decision','disc_dti','disc_worst',
                           'geo_dti','gain_vs_base','reasons']])
    print('Choice:', b['decision']['chosen'], '| Status:', bst.read_json('geology', 'status'))


## 9 — C: Zone robustness diagnostics and ablations

No zone-specific threshold tuning. Inspect weak-zone metric, input shifts and geographic preservation.

In [ ]:
if BOOST_RUN['robustness']:
    c = bst.c_robustness()
    for name in ('zone_ranking','composition','prediction_geometry'):
        display(c['diagnostics'][name])
    display(c['ablation'][['candidate','decision','disc_dti','disc_worst','geo_dti','reasons']])


## 10 — D: Fault-system hybrid recovery

Compare recovered and lost systems under the *same eligible prediction area*. Reject unacceptable false-positive increases.

In [ ]:
if BOOST_RUN['hybrid']:
    d = bst.d_hybrid()
    display(d['ablation'][['candidate','decision','disc_dti','disc_worst','geo_dti',
                           'systems_gained','systems_lost','fp_increase','reasons']])
    display(d['d3'])


## 11 — E: Candidate selection (export OFF by default)

Pass all provenance, parity, nested DTI, worst-zone and geographic gates before enabling export. Never upload automatically.

In [ ]:
if BOOST_RUN['selection']:
    e = bst.e_select(export=BOOST_RUN['export'])
    print(json.dumps({k: v for k, v in e.items() if k != 'champion'}, indent=2, default=str)[:5000])
    display(bst.load_table('candidates','selection'))
    display(bst.load_table('leaderboard_compare','selection'))
else:
    assert not BOOST_RUN['export'], 'Set selection=True before export=True'


## 12 — Reports, indexes, and saved artifacts

All outputs persist to Drive under `outputs/p6_fast_track/boost/`. Reports must never invent measurements for stages not yet run.

In [ ]:
if BOOST_RUN['report']:
    rep = bst.report()
    display(rep['status'])
    display(rep['summary'])
    print('Readiness:', json.dumps(rep['readiness'].get('questions', {}), indent=2, default=str)[:5000])
    print('Artifact directory:', p6.ws.root / 'boost')


## Execution decision checklist

1. First run: `preflight=True`, `benchmark=True`; all other compute stages OFF. Review preflight + A verdict.
2. Only if justified: enable `backfill_g02=True` (after inspecting plan), rerun preflight, enable geology.
3. When B has ACCEPT/REJECT: enable C and D individually; keep export OFF.
4. Enable E with export OFF; if all gates pass, rerun E with `export=True`. Upload submission **manually**.

**Baselines recorded (not measured by this notebook):** fallback discovery 0.161576, geographic 0.182832; W2 discovery 0.224640, worst zone 0.153414, geographic 0.180843.

**Source:** `docs/p6/P6_BOOST_HANDOFF.md`. This notebook is a new runner; repository r2 code must be available at selected Git ref.
